# Lab 02 — Binary Threat Detection

## Research question
Can baseline machine-learning models distinguish normal traffic from attack traffic?

## Hypothesis
A nonlinear model may outperform a linear baseline, but the useful comparison must include false positives and false negatives.

In [ ]:
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

# OPTIONAL — GOOGLE COLAB ONLY: clone the repo so src/ actually exists here
try:
    import google.colab
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()

# If running from the notebooks/ folder locally:
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
# OPTIONAL — GOOGLE COLAB ONLY
# Run this cell if the UNSW-NB15 CSV files are not already in DATA_DIR.

try:
    from google.colab import files

    if not (DATA_DIR / "UNSW_NB15_training-set.csv").exists():
        print(
            "Upload UNSW_NB15_training-set.csv, "
            "UNSW_NB15_testing-set.csv, and optionally UNSW_NB15_features.csv"
        )
        uploaded = files.upload()
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        for filename, content in uploaded.items():
            (DATA_DIR / filename).write_bytes(content)

except ImportError:
    print("Not running in Colab. Put the dataset CSV files in:", DATA_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import logistic_pipeline, random_forest_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [ ]:
logistic = logistic_pipeline(X_train)
logistic.fit(X_train, y_train)

pred_logistic = logistic.predict(X_test)
score_logistic = logistic.predict_proba(X_test)[:, 1]

logistic_metrics = binary_metrics(
    y_test,
    pred_logistic,
    score_logistic
)
display(binary_metrics_frame("Logistic Regression", logistic_metrics))

In [ ]:
forest = random_forest_pipeline(X_train)
forest.fit(X_train, y_train)

pred_forest = forest.predict(X_test)
score_forest = forest.predict_proba(X_test)[:, 1]

forest_metrics = binary_metrics(
    y_test,
    pred_forest,
    score_forest
)
display(binary_metrics_frame("Random Forest", forest_metrics))

In [ ]:
comparison = pd.concat([
    binary_metrics_frame("Logistic Regression", logistic_metrics),
    binary_metrics_frame("Random Forest", forest_metrics),
])
display(comparison.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_logistic, ax=axes[0], colorbar=False
)
axes[0].set_title("Logistic Regression")

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_forest, ax=axes[1], colorbar=False
)
axes[1].set_title("Random Forest")

plt.tight_layout()
plt.show()

## Your analysis

Explain:

- Which model catches more attacks?
- Which model generates fewer false alarms?
- Which has the lower false-negative rate?
- Why is accuracy alone insufficient?